# Build a Data Cleaning Helper

In the EDA lesson, we built a helper that answered questions about data. In this lesson we build one that fixes it. Data cleaning is an important step in the Data Science pipeline and let's see how we can utilise LLMs to do that.



## 1 - Setup

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


In [ ]:
%pip install -q google-genai pandas python-dotenv

In [8]:
import os
import re
import warnings
import pandas as pd
from google import genai
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

df = pd.read_csv("../../data/hr_analytics.csv")
print(f"Dataset: {df.shape}")

Dataset: (5030, 25)


Over view of the cleaning helper

![](../../images/cleaning_helper.png)

## 2 - The Data

**Some real problems in this dataset**

| Problem | Column(s) | Why it matters |
|---|---|---|
| Missing values | `manager_rating`, `distance_from_home`, `training_hours_last_year` | Imputation policy affects downstream modeling |
| Mixed date formats | `last_promotion_date` | Inconsistent parsing creates silent bugs |

In [9]:
# Missingness snapshot
df.isnull().sum()

Employee ID                   0
age                           0
gender                        0
department                    0
department_code               0
JobTitle                      0
job_level                     0
Education                     0
MonthlyIncome                 0
monthly_rate                  0
hourly_rate                   0
daily_rate                    0
years_at_company              0
years_in_role                 0
years_since_promotion         0
satisfaction_score            0
environment_satisfaction      0
Attrition                     0
OverTime                      0
distance_from_home           67
training_hours_last_year     64
num_companies_worked          0
manager_rating              460
work_life_balance             0
last_promotion_date          43
dtype: int64

In [10]:
# Mixed date formats
df["last_promotion_date"]

0       2024-09-06
1       12/18/2022
2       2017-12-04
3       2024-12-07
4       06/25/2021
           ...    
5025    04/15/2024
5026    09/11/2023
5027    2024-06-17
5028    2024-04-22
5029    07/21/2024
Name: last_promotion_date, Length: 5030, dtype: str

## 3 - Build the Profile

We will compute a lean statistical summary of the table so that the model gets a clearer view of what actually appears in the column.

In [11]:
def build_dataframe_profile(frame: pd.DataFrame) -> pd.DataFrame:
    """Build a compact profile of each column.

    Args:
        frame: Input DataFrame. Not modified in place.

    Returns:
        DataFrame with one row per column.
    """
    missing = frame.isna().sum()
    rows = []

    for col in frame.columns:
        series = frame[col]

        # top values with counts
        top_values = series.value_counts(dropna=True).head(5)
        top_values = [(str(v), int(c)) for v, c in top_values.items()]

        rows.append(
            {
                "column": col,
                "dtype": str(series.dtype),
                "missing": int(missing[col]),
                "missing_pct": round(missing[col] / len(frame) * 100, 1),
                "unique": int(series.nunique(dropna=True)),
                "top_values": top_values,
            }
        )

    return pd.DataFrame(rows)


def profile_to_str(profile_df: pd.DataFrame) -> str:
    """Convert profile DataFrame to a string for the LLM."""
    lines = []

    for _, row in profile_df.iterrows():
        lines.append(
            f"{row['column']} | dtype={row['dtype']} | "
            f"missing={row['missing']} ({row['missing_pct']}%) | "
            f"unique={row['unique']} | "
            f"top_values={row['top_values']}"
        )

    return "\n".join(lines)

In [12]:
build_dataframe_profile(df)

,column,dtype,missing,missing_pct,unique,top_values
0,Employee ID,int64,0,0.0,5000,"[(1108, 2), (1203, 2), (1265, 2), (1835, 2), (..."
1,age,int64,0,0.0,40,"[(21, 258), (33, 249), (29, 243), (31, 232), (..."
2,gender,str,0,0.0,3,"[(Male, 2429), (Female, 2356), (Non-Binary, 245)]"
3,department,str,0,0.0,6,"[(Engineering, 1433), (Sales, 1073), (Operatio..."
4,department_code,str,0,0.0,6,"[(ENG-02, 1433), (SAL-03, 1073), (OPS-06, 947)..."
5,JobTitle,str,0,0.0,42,"[(Logistics Coordinator, 165), (Supply Chain A..."
6,job_level,int64,0,0.0,5,"[(1, 1475), (2, 1447), (3, 1114), (4, 734), (5..."
7,Education,str,0,0.0,4,"[(Bachelor's, 2286), (Master's, 1380), (High S..."
8,MonthlyIncome,int64,0,0.0,3680,"[(6098, 6), (5155, 5), (6267, 5), (6674, 5), (..."
9,monthly_rate,int64,0,0.0,3889,"[(7630, 6), (10104, 5), (8910, 5), (8995, 5), ..."


## 4 - The Cleaning Helper

We define two prompts with two specific jobs. The diagnose step reasons about the data first and this catches more issues than going straight to code. You see the diagnosis before any code runs, so you can review and reject before implementation.
What the LLM does here: reads the profile, identifies every data quality issue, then writes the pandas code to fix them. You stay in control of what runs.

In [13]:
DIAGNOSE_PROMPT = (
    "You are a data quality expert. "
    "Given a dataset profile, identify every data quality issue you can find. "
    "For each issue state: the column, the problem, and the recommended fix. "
    "Consider: missing values, mixed formats, whitespace, inconsistent casing, "
    "identifier columns that should not be modified. "
    "Never use infer_datetime_format — removed in pandas 2.2. "
    "Return a structured list only. No code."
)

IMPLEMENT_PROMPT = (
    "You are a data-cleaning assistant. "
    "Write executable pandas code using the existing DataFrame variable `df`. "
    "Use up-to-date pandas 2.x and Python 3.10+ syntax. "
    "For numeric columns with low unique counts (rating scales), prefer mode over median for imputation. "
    "Never use infer_datetime_format — removed in pandas 2.2. "
    "For mixed date formats use pd.to_datetime(df[col], format='mixed'). "
    "Do not use inplace=True. Assign results back to columns. "
    "No imports. Save the final cleaned DataFrame to `result_df`. "
    "Return only executable Python code."
)

In [14]:
def diagnose_data(frame: pd.DataFrame):
    profile = build_dataframe_profile(frame)
    profile_str = profile_to_str(profile)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=profile_str,
        config={
            "temperature": 0.0,
            "seed": 42,
            "system_instruction": DIAGNOSE_PROMPT,
        },
    )
    return (response.text or "").strip()


In [15]:
diagnosis = diagnose_data(df)
print(diagnosis)

Here are the data quality issues identified from the dataset profile:

1.  **Column**: Employee ID
    *   **Problem**: The `Employee ID` column, which should uniquely identify each employee, contains duplicate values as indicated by `top_values` showing counts greater than 1 (e.g., '1108' appears 2 times). This suggests either duplicate records for the same employee or incorrect ID assignments.
    *   **Recommended Fix**: Investigate the source of duplicate `Employee ID` entries. If they represent duplicate rows for the same employee, remove the redundant rows, keeping the most complete or recent record. If they represent distinct employees assigned the same ID, correct the IDs to ensure uniqueness.

2.  **Column**: satisfaction_score
    *   **Problem**: The column is stored as a string (`dtype=str`) but contains both a numerical score and a descriptive text (e.g., '3 - High'). This mixed format makes it difficult to perform numerical analysis or sort by score directly.
    *   **Re

In [16]:
def generate_cleaning_code(frame: pd.DataFrame, diagnosis: str):
    profile = build_dataframe_profile(frame)
    profile_str = profile_to_str(profile)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"{profile_str}\n\nCleaning plan:\n{diagnosis}",
        config={
            "temperature": 0.0,
            "seed": 42,
            "system_instruction": IMPLEMENT_PROMPT,
        },
    )

    text = (response.text or "").strip()
    match = re.search(
        r"```(?:python)?\s*(.*?)```",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    return match.group(1).strip() if match else text


In [17]:
code = generate_cleaning_code(df, diagnosis)
print(code)

# Create a copy to work on, ensuring the original df is not modified
result_df = df.copy()

# 1. Employee ID: Remove duplicate Employee ID entries, keeping the first occurrence
result_df = result_df.drop_duplicates(subset=['Employee ID'], keep='first')

# 2. satisfaction_score: Extract numerical score and convert to integer
result_df['satisfaction_score'] = result_df['satisfaction_score'].str.extract(r'(\d+)').astype(int)

# 3. OverTime: Standardize casing and remove whitespace
result_df['OverTime'] = result_df['OverTime'].str.strip().str.title()

# 4. distance_from_home: Impute missing values with the median
median_distance = result_df['distance_from_home'].median()
result_df['distance_from_home'] = result_df['distance_from_home'].fillna(median_distance)

# 5. training_hours_last_year: Impute missing values with the median
median_training_hours = result_df['training_hours_last_year'].median()
result_df['training_hours_last_year'] = result_df['training_hours_last_year'].fillna(median_t

In [18]:
def apply_cleaning_code(frame: pd.DataFrame, code: str):
    env = {"pd": pd, "df": frame.copy()}
    exec(code, env, env)
    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result


In [19]:
df_clean = apply_cleaning_code(df, code)
df_clean.head()

,Employee ID,age,gender,department,department_code,JobTitle,job_level,Education,MonthlyIncome,monthly_rate,...,satisfaction_score,environment_satisfaction,Attrition,OverTime,distance_from_home,training_hours_last_year,num_companies_worked,manager_rating,work_life_balance,last_promotion_date
0,1001,27,Female,Engineering,ENG-02,Backend Developer,3,High School,8001,9162,...,3,4,No,No,11.0,32.0,1,4.0,Very High,2024-09-06
1,1002,34,Male,Engineering,ENG-02,Data Engineer,2,Master's,8777,10301,...,2,2,No,No,26.0,30.0,0,2.0,Low,2022-12-18
2,1003,50,Female,Finance,FIN-05,Controller,4,Master's,14021,18470,...,4,3,No,No,17.0,40.0,2,3.0,Medium,2017-12-04
3,1004,31,Male,Marketing,MKT-04,Marketing Analyst,3,Bachelor's,6402,8805,...,4,3,No,No,3.0,32.0,0,2.0,High,2024-12-07
4,1005,51,Male,Marketing,MKT-04,SEO Specialist,4,Master's,9277,12125,...,4,2,No,No,3.0,25.0,0,1.0,High,2021-06-25


In [20]:
df_clean.isnull().sum()

Employee ID                 0
age                         0
gender                      0
department                  0
department_code             0
JobTitle                    0
job_level                   0
Education                   0
MonthlyIncome               0
monthly_rate                0
hourly_rate                 0
daily_rate                  0
years_at_company            0
years_in_role               0
years_since_promotion       0
satisfaction_score          0
environment_satisfaction    0
Attrition                   0
OverTime                    0
distance_from_home          0
training_hours_last_year    0
num_companies_worked        0
manager_rating              0
work_life_balance           0
last_promotion_date         0
dtype: int64

## 5 - Wrapping all the modules in a single function


In [21]:
def cleaning_helper(
    frame: pd.DataFrame, show_code: bool = False, show_diagnosis: bool = False
):
    diagnosis = diagnose_data(frame)

    if show_diagnosis:
        print("--- Diagnosis ---")
        print(diagnosis)
        print()

    code = generate_cleaning_code(frame, diagnosis)

    if show_code:
        print("--- Generated Code ---")
        print(code)
        print("---\n")

    result_df = apply_cleaning_code(frame, code)
    return result_df, diagnosis, code


In [22]:
# run the cleaning helper
df_clean, diagnosis, code = cleaning_helper(df, show_code=True, show_diagnosis=True)

--- Diagnosis ---
Here are the data quality issues identified from the dataset profile:

1.  **Column**: Employee ID
    *   **Problem**: The `Employee ID` column, which should uniquely identify each employee, contains duplicate values as indicated by `top_values` showing counts greater than 1 (e.g., '1108' appears 2 times). This suggests either duplicate records for the same employee or incorrect ID assignments.
    *   **Recommended Fix**: Investigate the source of duplicate `Employee ID` entries. If they represent duplicate rows for the same employee, remove the redundant rows, keeping the most complete or recent record. If they represent distinct employees assigned the same ID, correct the IDs to ensure uniqueness.

2.  **Column**: satisfaction_score
    *   **Problem**: The column is stored as a string (`dtype=str`) but contains both a numerical score and a descriptive text (e.g., '3 - High'). This mixed format makes it difficult to perform numerical analysis or sort by score dire

In [23]:
# verify — shape preserved, missing values gone
print(f"Missing before: {df.isnull().sum().sum()}")
print(f"Missing after:  {df_clean.isnull().sum().sum()}")
print(f"last_promotion_date dtype: {df_clean['last_promotion_date'].dtype}")

Missing before: 634
Missing after:  0
last_promotion_date dtype: datetime64[us]


In [24]:
df_clean["last_promotion_date"].head()

0   2024-09-06
1   2022-12-18
2   2017-12-04
3   2024-12-07
4   2021-06-25
Name: last_promotion_date, dtype: datetime64[us]

## 6 - Edit and Re-run

If the generated code got something wrong, edit the specific line and re-run. No second LLM call needed for small fixes.
If the issue is bigger than a one-line fix, just re-run the helper with additional instructions appended to the profile



In [ ]:
# Editing LLM-generated code is as simple as a string replace.
# Example: swap any method the model got wrong before running it.
code_fixed = code.replace("<what the LLM wrote>", "<what you actually want>")

# e.g. code.replace("df['OverTime'].str.title()", "df['OverTime'].str.upper()")

env = {"pd": pd, "df": df.copy()}
exec(code_fixed, env, env)
env.get("result_df")

## 7 - Save for Reuse

The helper is saved to `cleaning_helper.py`. Lesson 2.2 imports it directly.


In [25]:
import inspect

components = [
    "import os",
    "import re",
    "import pandas as pd",
    "from google import genai",
    "from dotenv import load_dotenv",
    "",
    "load_dotenv()",
    "client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))",
    "",
    f"DIAGNOSE_PROMPT = {repr(DIAGNOSE_PROMPT)}",
    "",
    f"IMPLEMENT_PROMPT = {repr(IMPLEMENT_PROMPT)}",
    "",
    inspect.getsource(build_dataframe_profile),
    "",
    inspect.getsource(profile_to_str),
    "",
    inspect.getsource(diagnose_data),
    "",
    inspect.getsource(generate_cleaning_code),
    "",
    inspect.getsource(apply_cleaning_code),
    "",
    inspect.getsource(cleaning_helper),
]

with open("cleaning_helper.py", "w", encoding="utf-8") as f:
    f.write("\n".join(components))

print("Saved cleaning_helper.py")


Saved cleaning_helper.py
